# Transporter handoffs along the PT: a family-wide screen (notebook 101)

**The question.** Notebook 92 mapped one handoff along the continuous axis (glucose uptake, SGLT2 to SGLT1). Within
each solute-carrier and ABC family, do early (S1-high) and late (S3-high) members hand off more often than chance;
where along the axis do the handoffs sit; are they confirmed by microdissection and the human atlas; are they
conserved between species?

**The frozen protocol** is `docs/results/pt-transporter-handoffs.md` (commit 66afdb6).

**Sections.** 1 setup and inputs; 2 eligible genes and pairs; 3 Q1 enrichment; 4 profiles, crossings and widths;
5 Q2 confirmation; 6 Q3 conservation; 7 Q4 positions; 8 recovery checks; 9 figures; 10 reading.

## 1 · Setup and inputs

In [ ]:
import argparse
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
T71 = results_root / 'pt_pathways_package_pseudospace'
T90 = results_root / 'pt_continuous_axis_experiments' / 'tables'
HGNC = data_root / 'external' / 'hgnc'
CHEN = data_root / 'external' / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz'
CENSUS = data_root / 'external' / 'census_pt_segments'
ORTHOLOGS = results_root / 'minimal_pt_scfates' / 'ortholog_map_used.csv'
output = results_root / 'pt_transporter_handoffs'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_LOGIC_VERSION = '101.handoffs.1'
SEED = 101
MOUSE, HUMAN = ['Ctrl1A2', 'Ctrl1A4'], ['HUK1_COR1', 'HUK1_MED1']
SPECIES = {'mouse': MOUSE, 'human': HUMAN}
GRADED, N_NULL, INSIDE, CONFIRM_SHARE, MIN_CELLS = .5, 1000, .05, 2 / 3, 50
REQUIRED = [T90 / 'd_end_contrasts.csv.gz', T71 / 'tables' / 'universe_human_vs_mouse.csv', T71 / 'tables' / 'method_b_zones.csv',
            HGNC / 'hgnc_group_752.tsv', HGNC / 'hgnc_group_417.tsv', CHEN, ORTHOLOGS,
            CENSUS / 'human_pt_segment_pseudobulk_counts.csv.gz', CENSUS / 'human_pt_segment_pseudobulk_meta.csv']
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 71 and 90 first)')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from pseudospace.handoffs import crossing, family_null, half_points, pair_count, scaled_profile

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
SPECIES_COLOR = {'mouse': '#0072B2', 'human': '#D55E00'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 60, 'display.max_rows', 120)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


def stored(folder, stage):
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


guard_log = []
END = pd.read_csv(T90 / 'd_end_contrasts.csv.gz').set_index('mouse')
END.index = END.index.astype(str)
U71 = pd.read_csv(T71 / 'tables' / 'universe_human_vs_mouse.csv').gene.astype(str).tolist()
payload = stored(T71, 'gene_model_human_vs_mouse_species')
GRID = payload['grid'].astype(float)
CURVES = {n: pd.DataFrame(payload[f'curve|{n}'], index=U71) for n in MOUSE + HUMAN}
assert all(c.shape == (len(U71), len(GRID)) for c in CURVES.values()), 'G1: curve shapes'
zones = pd.read_csv(T71 / 'tables' / 'method_b_zones.csv').set_index('dataset').loc['Human vs mouse PT']
CUTS = (float(zones['S1|S2 cut']), float(zones['S2|S3 cut']))
guard_log.append(f"G1: 71's human vs mouse species curves for {len(U71):,} genes on a {len(GRID)}-point grid "
                 f"({GRID[0]:.3f}–{GRID[-1]:.3f}); cuts {CUTS[0]:.3f}, {CUTS[1]:.3f}")
orth = pd.read_csv(ORTHOLOGS)
h2m = dict(zip(orth.human_symbol.astype(str), orth.mouse_symbol.astype(str)))
hg = pd.concat([pd.read_csv(HGNC / f, sep='\t') for f in ('hgnc_group_752.tsv', 'hgnc_group_417.tsv')])
hg = hg[hg['Approved symbol'].isin(h2m)]
FAMILY = hg.drop_duplicates('Approved symbol').assign(mouse=lambda d: d['Approved symbol'].map(h2m)).set_index('mouse')['Group name']
print('\n'.join('· ' + g for g in guard_log))
print(f'{len(FAMILY):,} transporters with a one-to-one mouse ortholog in {FAMILY.nunique()} families')

## 2 · Eligible genes, early and late, pairs

In [ ]:
ELIG, CLASS = {}, {}
rows = []
for sp, specs in SPECIES.items():
    ok_curve = np.all([np.isfinite(CURVES[s].reindex(FAMILY.index)).any(axis=1) for s in specs], axis=0)
    genes = [g for g, ok in zip(FAMILY.index, ok_curve) if ok and g in END.index]
    e = END.loc[genes, [f'E {s}' for s in specs]]
    early = (e <= -GRADED).all(axis=1)
    late = (e >= GRADED).all(axis=1)
    ELIG[sp] = genes
    CLASS[sp] = pd.Series(np.where(early, 'early', np.where(late, 'late', 'neither')), index=genes)
ELIGIBLE = pd.DataFrame({'family': FAMILY, **{f'class {sp}': CLASS[sp] for sp in SPECIES},
                         **{f'E {s}': END[f'E {s}'].reindex(FAMILY.index) for s in MOUSE + HUMAN},
                         'fold-protected': END['fold-protected'].reindex(FAMILY.index)})
ELIGIBLE = ELIGIBLE[ELIGIBLE[[f'class {sp}' for sp in SPECIES]].notna().any(axis=1)]
ELIGIBLE.to_csv(tables / 'eligible.csv')
display(pd.DataFrame({sp: CLASS[sp].value_counts() for sp in SPECIES}))

PAIRS = []
for sp in SPECIES:
    c = CLASS[sp]
    fam = FAMILY.reindex(c.index)
    for f, members in c.groupby(fam):
        for a in members.index[members.eq('early')]:
            for b in members.index[members.eq('late')]:
                PAIRS.append({'species': sp, 'family': f, 'early': a, 'late': b})
PAIRS = pd.DataFrame(PAIRS)
print({sp: int((PAIRS.species == sp).sum()) for sp in SPECIES}, 'handoff pairs')

## 3 · Q1 · Are handoffs enriched within families?

In [ ]:
Q1 = []
NULLS = {}
for sp in SPECIES:
    c = CLASS[sp]
    fam = FAMILY.reindex(c.index).to_numpy()
    obs = pair_count(fam, c.eq('early'), c.eq('late'))
    null = family_null(fam, c.eq('early').to_numpy(), c.eq('late').to_numpy(), n=N_NULL, seed=SEED)
    NULLS[sp] = null
    all_e = END[[f'E {s}' for s in SPECIES[sp]]]
    Q1.append({'species': sp, 'eligible transporters': len(c), 'families': int(pd.Series(fam).nunique()),
               'early': int(c.eq('early').sum()), 'late': int(c.eq('late').sum()), 'pairs': obs,
               'null mean': float(null.mean()), 'null 95%': float(np.quantile(null, .95)),
               'p': float((1 + (null >= obs).sum()) / (1 + N_NULL)),
               'families with a handoff': int(PAIRS[PAIRS.species.eq(sp)].family.nunique()),
               'graded share, transporters': float(c.ne('neither').mean()),
               'graded share, all genes': float(((all_e <= -GRADED).all(axis=1) | (all_e >= GRADED).all(axis=1)).mean())})
Q1 = pd.DataFrame(Q1).set_index('species')
Q1.to_csv(tables / 'q1_enrichment.csv')
display(Q1.round(3))

## 4 · Profiles, crossings and widths

In [ ]:
def species_profile(gene, sp):
    prof = [scaled_profile(CURVES[s].loc[gene].to_numpy(float) / np.log(2)) for s in SPECIES[sp]]
    return np.nanmean(np.vstack(prof), axis=0)


rows = []
for r in PAIRS.itertuples():
    for sp in SPECIES:
        if r.early not in CURVES[SPECIES[sp][0]].index:
            continue
        pe, pl = species_profile(r.early, sp), species_profile(r.late, sp)
        fall, rise = half_points(pe, pl, GRID)
        rows.append({'pair species': r.species, 'family': r.family, 'early': r.early, 'late': r.late, 'profile species': sp,
                     'x*': crossing(pe, pl, GRID), 'early 50% fall': fall, 'late 50% rise': rise, 'width': rise - fall})
POS = pd.DataFrame(rows)

## 5 · Q2 · Microdissection and atlas confirmation

In [ ]:
chen = pd.read_csv(CHEN, sep='\t').drop_duplicates('Gene_symbol').set_index('Gene_symbol')
s1 = chen[[c for c in chen.columns if '_PTS1_' in c]].mean(axis=1)
s3 = chen[[c for c in chen.columns if '_PTS3_' in c]].mean(axis=1)
CHEN_RATIO = np.log2((s1 + 1) / (s3 + 1))
cmeta = pd.read_csv(CENSUS / 'human_pt_segment_pseudobulk_meta.csv', index_col=0)
ccounts = pd.read_csv(CENSUS / 'human_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
S3_TYPE, PCT_TYPE = 'epithelial cell of proximal tubule segment 3', 'kidney proximal convoluted tubule epithelial cell'
donors = [d for d, b in cmeta.groupby('donor_id') if {S3_TYPE, PCT_TYPE} <= set(b.cell_type[b.n_cells >= MIN_CELLS])]
diffs = []
for d in donors:
    b = cmeta[cmeta.donor_id.eq(d)]
    v = {}
    for ct in (S3_TYPE, PCT_TYPE):
        rows_ = b.index[b.cell_type.eq(ct)]
        y = ccounts.loc[rows_].sum(axis=0)
        v[ct] = np.log2((y + .5) / (b.loc[rows_, 'total_counts_all_genes'].sum() + 1) * 1e6)
    diffs.append(v[PCT_TYPE] - v[S3_TYPE])
CENSUS_RATIO = pd.concat(diffs, axis=1).mean(axis=1)
CENSUS_RATIO.index = CENSUS_RATIO.index.astype(str)
PAIRS['Chen early'] = CHEN_RATIO.reindex(PAIRS.early).to_numpy()
PAIRS['Chen late'] = CHEN_RATIO.reindex(PAIRS.late).to_numpy()
PAIRS['Census early'] = CENSUS_RATIO.reindex(PAIRS.early).to_numpy()
PAIRS['Census late'] = CENSUS_RATIO.reindex(PAIRS.late).to_numpy()
PAIRS['confirmed'] = np.where(PAIRS.species.eq('mouse'), (PAIRS['Chen early'] > 0) & (PAIRS['Chen late'] < 0),
                              (PAIRS['Census early'] > 0) & (PAIRS['Census late'] < 0))
PAIRS['testable'] = np.where(PAIRS.species.eq('mouse'), PAIRS[['Chen early', 'Chen late']].notna().all(axis=1),
                             PAIRS[['Census early', 'Census late']].notna().all(axis=1))
Q2 = PAIRS[PAIRS.testable].groupby('species').confirmed.agg(['size', 'sum', 'mean']).rename(
    columns={'size': 'testable pairs', 'sum': 'confirmed', 'mean': 'share'})
Q2['holds'] = Q2.share >= CONFIRM_SHARE
Q2.to_csv(tables / 'q2_confirmation.csv')
display(Q2.round(3))
print(f'Census donors: {len(donors)}')

## 6 · Q3 · Conservation between species

In [ ]:
key = lambda d: set(zip(d.family, d.early, d.late))  # noqa: E731
m_pairs, h_pairs = key(PAIRS[PAIRS.species.eq('mouse')]), key(PAIRS[PAIRS.species.eq('human')])
conserved = m_pairs & h_pairs
reversed_ = {(f, a, b) for f, a, b in m_pairs if (f, b, a) in h_pairs}
both = sorted(set(ELIG['mouse']) & set(ELIG['human']))
fam_both = FAMILY.reindex(both)
n_candidates = int(sum(n * (n - 1) for n in fam_both.value_counts()))
rate = {sp: len([p for p in (m_pairs if sp == 'mouse' else h_pairs) if p[1] in both and p[2] in both]) / max(n_candidates, 1)
        for sp in SPECIES}
Q3 = pd.DataFrame([{'genes eligible in both': len(both), 'ordered within-family candidate pairs': n_candidates,
                    'mouse pairs (both eligible)': int(rate['mouse'] * n_candidates),
                    'human pairs (both eligible)': int(rate['human'] * n_candidates),
                    'conserved': len(conserved), 'expected by chance': rate['mouse'] * rate['human'] * n_candidates,
                    'reversed': len(reversed_)}])
Q3.to_csv(tables / 'q3_conservation.csv', index=False)
display(Q3.round(2))
print('conserved:', sorted(conserved))
print('reversed:', sorted(reversed_))

## 7 · Q4 · Where the handoffs sit

In [ ]:
def inside(x):
    return np.isfinite(x) & (np.abs(x - CUTS[0]) > INSIDE) & (np.abs(x - CUTS[1]) > INSIDE)


own = POS[POS['pair species'].eq(POS['profile species'])].copy()
rng = np.random.default_rng(SEED)
RAND = []
for sp in SPECIES:
    e_all = END[[f'E {s}' for s in SPECIES[sp]]]
    pool_e = [g for g in e_all.index[(e_all <= -GRADED).all(axis=1)] if g in CURVES[SPECIES[sp][0]].index and g not in FAMILY.index]
    pool_l = [g for g in e_all.index[(e_all >= GRADED).all(axis=1)] if g in CURVES[SPECIES[sp][0]].index and g not in FAMILY.index]
    for _ in range(N_NULL):
        a, b = rng.choice(pool_e), rng.choice(pool_l)
        pe, pl = species_profile(a, sp), species_profile(b, sp)
        fall, rise = half_points(pe, pl, GRID)
        RAND.append({'species': sp, 'x*': crossing(pe, pl, GRID), 'width': rise - fall})
RAND = pd.DataFrame(RAND)
Q4 = []
for sp in SPECIES:
    o, r = own[own['pair species'].eq(sp)], RAND[RAND.species.eq(sp)]
    Q4.append({'species': sp, 'pairs': len(o), 'median x*': o['x*'].median(), 'share inside a segment': float(inside(o['x*'].to_numpy()).mean()),
               'median width': o.width.median(), 'share overlapping (width < 0)': float((o.width < 0).mean()),
               'random pairs: median x*': r['x*'].median(), 'random: share inside': float(inside(r['x*'].to_numpy()).mean()),
               'random: median width': r.width.median()})
Q4 = pd.DataFrame(Q4).set_index('species')
Q4.to_csv(tables / 'q4_positions.csv')
display(Q4.round(3))
POS.to_csv(tables / 'pair_positions.csv', index=False)
PAIRS.merge(own[['pair species', 'early', 'late', 'x*', 'width']].rename(columns={'pair species': 'species'}),
            on=['species', 'early', 'late'], how='left').to_csv(tables / 'pairs.csv', index=False)

## 8 · Recovery checks

In [ ]:
def is_pair(sp, a, b):
    return bool(((PAIRS.species == sp) & (PAIRS.early == a) & (PAIRS.late == b)).any())


REC = []
for name, a, b, species in (('R1 GLUT2 → GLUT1', 'Slc2a2', 'Slc2a1', ['mouse', 'human']),
                            ('R2 SGLT2 → SGLT1 (human)', 'Slc5a2', 'Slc5a1', ['human']),
                            ('R3 SGLT2 → SGLT1 (mouse, descriptive)', 'Slc5a2', 'Slc5a1', ['mouse'])):
    for sp in species:
        REC.append({'check': name, 'species': sp, 'handoff pair': is_pair(sp, a, b),
                    'E early (both specimens)': ', '.join(f"{END.loc[a, f'E {s}']:.2f}" for s in SPECIES[sp]) if a in END.index else 'absent',
                    'E late (both specimens)': ', '.join(f"{END.loc[b, f'E {s}']:.2f}" for s in SPECIES[sp]) if b in END.index else 'absent',
                    'fold-protected (early, late)': f"{END['fold-protected'].get(a)}, {END['fold-protected'].get(b)}"})
REC = pd.DataFrame(REC)
REC.to_csv(tables / 'recovery_checks.csv', index=False)
display(REC)

## 9 · Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(170 * MM, 90 * MM), sharex=True)
for ax, sp in zip(axes, SPECIES):
    o = own[own['pair species'].eq(sp)].sort_values('x*').reset_index(drop=True)
    for i, r in o.iterrows():
        lo, hi = sorted([r['early 50% fall'], r['late 50% rise']])
        ax.plot([lo, hi], [i, i], color=SPECIES_COLOR[sp] if r.width >= 0 else MUTED, lw=1.5)
        ax.plot(r['x*'], i, '|', color=INK, ms=4)
        ax.annotate(f"{r.early}→{r.late}", (max(hi, r['x*']) + .01, i), fontsize=4, va='center')
    for c in CUTS:
        ax.axvline(c, color=MUTED, lw=.5, ls=':')
    ax.set_yticks([])
    ax.set_title(sp)
    ax.set_xlabel('PT position')
    ax.set_xlim(GRID[0], GRID[-1] + .15)
save(fig, '101a_handoff_ladder')
plt.show()

show = [('Slc2a2', 'Slc2a1'), ('Slc5a2', 'Slc5a1')]
top = own.assign(conf=own.merge(PAIRS, left_on=['pair species', 'early', 'late'], right_on=['species', 'early', 'late'], how='left').confirmed.to_numpy())
for _, r in top[top.conf.fillna(False)].sort_values('width').head(4).iterrows():
    if (r.early, r.late) not in show:
        show.append((r.early, r.late))
fig, axes = plt.subplots(2, len(show), figsize=(40 * MM * len(show), 60 * MM), sharex=True, sharey=True, squeeze=False)
for j, (a, b) in enumerate(show):
    for i, sp in enumerate(SPECIES):
        ax = axes[i, j]
        if a in CURVES[SPECIES[sp][0]].index and b in CURVES[SPECIES[sp][0]].index:
            ax.plot(GRID, species_profile(a, sp), color=SPECIES_COLOR[sp], lw=1)
            ax.plot(GRID, species_profile(b, sp), color=SPECIES_COLOR[sp], lw=1, ls='--')
        for c in CUTS:
            ax.axvline(c, color=MUTED, lw=.5, ls=':')
        ax.set_title(f'{a} → {b} · {sp}', fontsize=5.5)
for ax in axes[-1]:
    ax.set_xlabel('PT position')
save(fig, '101b_handoff_profiles')
plt.show()

both_pos = POS[POS['pair species'].eq('mouse')].pivot_table(index=['early', 'late'], columns='profile species', values='x*')
fig, ax = plt.subplots(figsize=(60 * MM, 55 * MM))
if {'mouse', 'human'} <= set(both_pos.columns):
    ax.scatter(both_pos['mouse'], both_pos['human'], s=8, color=INK)
ax.plot([GRID[0], GRID[-1]], [GRID[0], GRID[-1]], color=MUTED, lw=.5, ls=':')
ax.set_xlabel('crossing x*, mouse'); ax.set_ylabel('crossing x*, human')
save(fig, '101c_crossing_mouse_human')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(110 * MM, 40 * MM))
for ax, sp in zip(axes, SPECIES):
    ax.hist(NULLS[sp], bins=30, color=LIGHT)
    ax.axvline(Q1.loc[sp, 'pairs'], color=SPECIES_COLOR[sp])
    ax.set_title(sp); ax.set_xlabel('within-family handoff pairs')
save(fig, '101d_q1_null')
plt.show()

## 10 · How the protocol was read, and the run record

In [ ]:
READING = pd.DataFrame([
    ('reading', 'A transporter in more than one HGNC group keeps its first group (SLC file first, then ABC).'),
    ('reading', "Chen's ratio uses the mean TPM over its PTS1 and PTS3 replicate columns; Census uses donors with ≥ 50 cells of both types."),
    ('reading', 'Random comparison pairs for Q4 draw an early and a late non-transporter gene of the same species (1,000 pairs).'),
], columns=['kind', 'what'])
READING.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
RUN = {'notebook': '101_pt_transporter_handoffs', 'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED, 'guards': guard_log,
       'q1': Q1[['pairs', 'null mean', 'p']].round(4).to_dict('index'), 'q2': Q2.round(3).to_dict('index'),
       'conserved': sorted(map(list, conserved)), 'reversed': sorted(map(list, reversed_)),
       'recovery': REC[['check', 'species', 'handoff pair']].to_dict('records'),
       'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(RUN, indent=2, default=str))
print(json.dumps(RUN, indent=2, default=str))